# JUCKEX Quantum — 무료 GPU 학습 실험실

이 노트북은 **무료 Google Colab 세션**에서 승인된 한국어 위키백과 문서를 내려받아 작은 Quantum Transformer 후보를 처음부터 학습합니다. 유료 런타임이나 유료 저장소는 사용하지 않습니다.

- Cloudflare Worker의 `/export-approved` 경로가 배포되어 있어야 합니다.
- 관리자 인증 토큰과 승인된 문서 10개 이상이 필요합니다.
- 무료 GPU는 사용 가능 여부·시간·할당량이 보장되지 않습니다. 세션이 끊기면 다시 실행해야 합니다.
- 학습 결과는 후보 체크포인트일 뿐이며, 서비스 모델을 자동으로 바꾸지 않습니다.
- 위키백과 CC BY-SA 조건에 따라 출처·라이선스·동일조건변경허락 의무를 유지해야 합니다.


In [ ]:
!git clone --depth 1 --branch quantum-v2-development https://github.com/DEV1ND/BC-COUNTRY-.git /content/BC-COUNTRY-
%cd /content/BC-COUNTRY-
!pip -q install requests
import os, sys, json, time, random, getpass, hashlib, requests
import torch
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))
sys.path.append('/content/BC-COUNTRY-/quantum-inference')
from bpe_tokenizer import BPETokenizer
from google.colab import files


In [ ]:
WORKER_URL = input('Quantum Wikipedia Worker URL (기본값 사용 가능): ').strip() or 'https://juckex-quantum-wikipedia-explorer.bcsite.workers.dev'
ADMIN_TOKEN = getpass.getpass('Worker ADMIN_TOKEN: ').strip()
assert WORKER_URL.startswith('https://'), 'HTTPS 주소를 입력하세요.'
assert ADMIN_TOKEN, '관리자 토큰이 필요합니다.'
documents, cursor = [], ''
for page in range(100):  # 한 세션에서 최대 1,000개 문서
    response = requests.get(WORKER_URL.rstrip('/') + '/export-approved',
        params={'limit':10, 'after':cursor},
        headers={'Authorization':'Bearer ' + ADMIN_TOKEN}, timeout=40)
    response.raise_for_status()
    payload = response.json()
    documents.extend(payload.get('documents', []))
    next_cursor = payload.get('next_after')
    if not payload.get('has_more') or not next_cursor or next_cursor == cursor: break
    cursor = next_cursor
print('가져온 승인 문서:', len(documents))
documents = [d for d in documents if len(d.get('text','').strip()) >= 1000]
assert len(documents) >= 10, f'학습에는 승인된 유효 문서가 10개 이상 필요합니다. 현재 {len(documents)}개입니다.'
print('유효 문서:', len(documents), '| 예시:', documents[0]['title'])


In [ ]:
random.seed(20261121)
documents = sorted(documents, key=lambda d: hashlib.sha256(d['page_id'].encode()).hexdigest())
split = max(1, min(len(documents)-1, int(len(documents)*0.9)))
train_docs, valid_docs = documents[:split], documents[split:]
# 한 페이지 최대 20,000자, 전체 학습 텍스트 최대 약 2백만 자로 제한해 무료 세션 부담을 줄입니다.
def clean_docs(rows):
    out=[]
    for d in rows:
        text=' '.join(str(d.get('text','')).split())[:20000]
        if len(text) >= 1000: out.append((d,text))
    return out
train_docs, valid_docs = clean_docs(train_docs), clean_docs(valid_docs)
assert len(train_docs) >= 8 and len(valid_docs) >= 1, '학습/검증 문서가 부족합니다.'
train_texts=[t for _,t in train_docs]
valid_texts=[t for _,t in valid_docs]
print('학습 페이지:',len(train_texts),'| 검증 페이지:',len(valid_texts))
tokenizer=BPETokenizer.train(train_texts, vocab_size=1024, min_pair_frequency=2, extra_alphabet_texts=valid_texts)
print('Tokenizer vocab:',len(tokenizer.vocab))
BLOCK=256
def make_chunks(texts):
    chunks=[]
    for text in texts:
        ids=tokenizer.encode(text)
        for start in range(0, len(ids)-BLOCK, BLOCK):
            part=ids[start:start+BLOCK+1]
            if len(part)==BLOCK+1: chunks.append(part)
    return chunks
train_chunks=make_chunks(train_texts)
valid_chunks=make_chunks(valid_texts)
assert len(train_chunks)>=8 and len(valid_chunks)>=1, '텍스트 청크가 부족합니다. 더 많은 승인 페이지를 모아주세요.'
print('Train chunks:',len(train_chunks),'| Validation chunks:',len(valid_chunks))


In [ ]:
import torch.nn as nn
import torch.nn.functional as F
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
class QuantumBootstrap(nn.Module):
    def __init__(self,vocab,width=64,heads=4,layers=2,block=256):
        super().__init__(); self.block=block
        self.token=nn.Embedding(vocab,width); self.position=nn.Embedding(block,width)
        layer=nn.TransformerEncoderLayer(d_model=width,nhead=heads,dim_feedforward=width*4,
            dropout=0.1,batch_first=True,norm_first=True,activation='gelu')
        self.encoder=nn.TransformerEncoder(layer,num_layers=layers)
        self.norm=nn.LayerNorm(width); self.output=nn.Linear(width,vocab)
        self.register_buffer('causal',torch.triu(torch.ones(block,block,dtype=torch.bool),diagonal=1))
    def forward(self,ids):
        n=ids.shape[1]; x=self.token(ids)+self.position(torch.arange(n,device=ids.device))
        return self.output(self.norm(self.encoder(x,mask=self.causal[:n,:n])))
ARCH={'width':64,'heads':4,'layers':2,'block':BLOCK}
model=QuantumBootstrap(len(tokenizer.vocab),**ARCH).to(DEVICE)
print('Parameters:',sum(p.numel() for p in model.parameters()),'| device:',DEVICE)
@torch.no_grad()
def evaluate():
    model.eval(); total=0.0; count=0
    for part in valid_chunks:
        x=torch.tensor(part[:-1],device=DEVICE).unsqueeze(0)
        y=torch.tensor(part[1:],device=DEVICE)
        total += F.cross_entropy(model(x)[0],y,reduction='sum').item(); count += len(y)
    return total/max(1,count)
baseline=evaluate(); print('Initial validation loss:',round(baseline,4))


In [ ]:
STEPS=500  # 무료 세션이 짧으면 100~200으로 낮추세요.
BATCH=8 if torch.cuda.is_available() else 2
optimizer=torch.optim.AdamW(model.parameters(),lr=3e-4,weight_decay=0.01)
best_loss=float('inf'); best_state=None; history=[]
for step in range(1,STEPS+1):
    batch=random.choices(train_chunks,k=BATCH)
    x=torch.tensor([p[:-1] for p in batch],dtype=torch.long,device=DEVICE)
    y=torch.tensor([p[1:] for p in batch],dtype=torch.long,device=DEVICE)
    model.train(); optimizer.zero_grad(set_to_none=True)
    loss=F.cross_entropy(model(x).reshape(-1,len(tokenizer.vocab)),y.reshape(-1))
    loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); optimizer.step()
    if step % 50 == 0 or step == STEPS:
        val=evaluate(); history.append({'step':step,'train_loss':float(loss.item()),'validation_loss':val})
        print(f'Step {step}/{STEPS} | train {loss.item():.4f} | validation {val:.4f}')
        if val < best_loss:
            best_loss=val; best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
assert best_state is not None, '검증 체크포인트를 만들지 못했습니다.'
model.load_state_dict(best_state)
improved=best_loss < baseline
report={'status':'candidate_ready_for_review' if improved else 'needs_more_data_or_training',
 'created_at':int(time.time()),'documents':len(documents),'train_pages':len(train_docs),
 'validation_pages':len(valid_docs),'train_chunks':len(train_chunks),'validation_chunks':len(valid_chunks),
 'steps':STEPS,'initial_validation_loss':baseline,'best_validation_loss':best_loss,
 'validation_improved':improved,'architecture':ARCH,'vocab_size':len(tokenizer.vocab),
 'source':'Korean Wikipedia','license':'CC BY-SA 4.0; preserve attribution and share-alike terms',
 'serving_model_changed':False,'warning':'Next-token loss is not factual accuracy or conversational quality.'}
torch.save({'state_dict':model.state_dict(),'tokenizer':tokenizer.to_dict(),
 'architecture':ARCH,'steps':STEPS,'training_metadata':report},'/content/bpe_candidate_checkpoint.pt')
with open('/content/quantum_training_report.json','w',encoding='utf-8') as f: json.dump(report,f,ensure_ascii=False,indent=2)
print(json.dumps(report,ensure_ascii=False,indent=2))


In [ ]:
print('후보 모델을 검토한 뒤 별도 실험 환경에만 연결하세요. 운영 모델에는 자동 반영되지 않습니다.')
files.download('/content/bpe_candidate_checkpoint.pt')
files.download('/content/quantum_training_report.json')
